In [1]:
import torch

In [92]:
class Test:

    FITNESS_DONATION = 0.2  # Percentage of fitness donated to the neighbor
    FITNESS_DONATION_BONUS = 0.0
    EPS = 1e-6
    COLONIZE_PROB_ONE = 2 # 2 fitness for prob 100% of colonization
    REWARD_FOR_DONE = 1.0  # Reward for reaching the done condition
    FITNESS_GROWTH_VALUE = 0.01 # number of fitness points gained at every step

    def __init__(self, grid):
        self.grid = grid


    def manage_donations(self, action_grid):
        # 0. shorthand
        B, P, H, W = self.grid.shape
        EPS = self.EPS
        DON = self.FITNESS_DONATION                 # 0 < DON ≤ 1, e.g. 0.10

        # 1. prepare action mask (B,1,H,W) that broadcasts over populations 
        shifted_action = (action_grid - 1).unsqueeze(1)         # -1 means “no donation”

        # helper: pick donors for one direction, roll them to the neighbour
        def roll_from_dir(direction: int, shift: tuple[int,int]) -> torch.Tensor:
            donors = torch.where(shifted_action == direction, self.grid, 0.0)  # (B,P,H,W)
            return torch.roll(donors, shifts=shift, dims=(2, 3))

        #  2. donations sent to the eight neighbours (B,P,H,W)
        contrib = (
            roll_from_dir(0, (-1,  0))   # up
            + roll_from_dir(1, (-1,  1))   # up-right
            + roll_from_dir(2, ( 0,  1))   # right
            + roll_from_dir(3, ( 1,  1))   # down-right
            + roll_from_dir(4, ( 1,  0))   # down
            + roll_from_dir(5, ( 1, -1))   # down-left
            + roll_from_dir(6, ( 0, -1))   # left
            + roll_from_dir(7, (-1, -1))   # up-left
        ) * DON                                            # scale by the donation %

        #  3. update fitness of the receivers
        new_possible_grid = self.grid + contrib                     # every pop keeps its identity

        #  5. handle cells that were completely empty before the step ---------------
        #     • detect them using the *old* grid
        #     • keep only the population that now has the largest value
        empty_mask = (self.grid <= EPS).all(dim=1, keepdim=True)    # (B,1,H,W)

        # skip if none empty
        # winner population in each empty cell
        _, argmax_pop = new_possible_grid.max(dim=1)                     # (B,H,W)
        one_hot = torch.nn.functional.one_hot(argmax_pop, num_classes=P) \
                    .permute(0,3,1,2).bool()                   # (B,P,H,W)

        actual_donnations = torch.where(empty_mask & one_hot, contrib, 0.0)  # (B,P,H,W)

        new_grid = self.grid + actual_donnations

        #  4. if the *source* cell donated, take the 10 % penalty --------------------
        is_donor = (shifted_action >= 0) & (shifted_action <= 7)   # (B,1,H,W)
        new_grid = torch.where(is_donor, new_grid * (1 - DON), new_grid)

        #  6. clamp negatives that might arise from numerical noise -----------------
        self.grid = torch.clamp(new_grid, min=0.0)

In [94]:
grid = torch.tensor(
        [[[[5., 0., 0.],
           [0., 10., 0.],
           [0., 0., 0.]],          # pop 0
          [[0., 0., 0.],
           [0., 0., 0.],
           [1., 1., 3.]]],
           [[[1., 0., 0.],
           [0., 5., 0.],
           [0., 0., 0.]],          # pop 0
          [[0., 0., 0.],
           [0., 0., 0.],
           [1., 0., 3.]]]]         # pop 1
    )

# action grid (B=1, H=W=3)
#
# • (0,0) has value 5 and donates **down-right** → code 4
# • (1,1) has value 10 and donates **right**     → code 3
# everything else: 0 = no donation
#
action_grid = torch.tensor([[[3, 0, 0],
                            [0, 1, 0],
                            [1, 1, 6]], [[3, 0, 0],
                            [0, 1, 0],
                            [1, 1, 6]]], dtype=torch.long)


test = Test(grid)
test.grid

tensor([[[[ 5.,  0.,  0.],
          [ 0., 10.,  0.],
          [ 0.,  0.,  0.]],

         [[ 0.,  0.,  0.],
          [ 0.,  0.,  0.],
          [ 1.,  1.,  3.]]],


        [[[ 1.,  0.,  0.],
          [ 0.,  5.,  0.],
          [ 0.,  0.,  0.]],

         [[ 0.,  0.,  0.],
          [ 0.,  0.,  0.],
          [ 1.,  0.,  3.]]]])

In [95]:
test.manage_donations(action_grid)
print(test.grid)

tensor([[[[4.0000, 3.0000, 0.0000],
          [0.0000, 8.0000, 0.0000],
          [0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000],
          [0.2000, 0.0000, 0.0000],
          [0.8000, 0.8000, 2.4000]]],


        [[[0.8000, 1.2000, 0.0000],
          [0.0000, 4.0000, 0.0000],
          [0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000],
          [0.2000, 0.0000, 0.0000],
          [0.8000, 0.0000, 2.4000]]]])


In [44]:
grid = torch.tensor(
        [[[[5., 0., 0.],
           [0., 10., 0.],
           [0., 0., 0.]],          # pop 0
          [[0., 0., 0.],
           [0., 0., 0.],
           [5., 0., 3.]]]]         # pop 1
    )

# action grid (B=1, H=W=3)
#
# • (0,0) has value 5 and donates **down-right** → code 4
# • (1,1) has value 10 and donates **right**     → code 3
# everything else: 0 = no donation
#
action_grid = torch.tensor([[[3, 0, 0],
                            [0, 3, 0],
                            [3, 0, 1]]], dtype=torch.long)

# 0. shorthand
B, P, H, W = grid.shape
EPS = 1e-6
DON = 0.1                 # 0 < DON ≤ 1, e.g. 0.10

# 1. prepare action mask (B,1,H,W) that broadcasts over populations 
shifted_action = (action_grid - 1).unsqueeze(1)         # -1 means “no donation”

# helper: pick donors for one direction, roll them to the neighbour
def roll_from_dir(direction: int, shift: tuple[int,int]) -> torch.Tensor:
    donors = torch.where(shifted_action == direction, grid, 0.0)  # (B,P,H,W)
    return torch.roll(donors, shifts=shift, dims=(2, 3))

#  2. donations sent to the eight neighbours (B,P,H,W)
contrib = (
    roll_from_dir(0, (-1,  0))   # up
    + roll_from_dir(1, (-1,  1))   # up-right
    + roll_from_dir(2, ( 0,  1))   # right
    + roll_from_dir(3, ( 1,  1))   # down-right
    + roll_from_dir(4, ( 1,  0))   # down
    + roll_from_dir(5, ( 1, -1))   # down-left
    + roll_from_dir(6, ( 0, -1))   # left
    + roll_from_dir(7, (-1, -1))   # up-left
) * DON                                            # scale by the donation %

#  3. update fitness of the receivers

new_grid = grid + contrib                     # every pop keeps its identity

contrib                     # every pop keeps its identity

tensor([[[[0.0000, 0.5000, 0.0000],
          [0.0000, 0.0000, 1.0000],
          [0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.3000],
          [0.0000, 0.5000, 0.0000]]]])

In [45]:
is_donor = (shifted_action >= 0) & (shifted_action <= 7)   # (B,1,H,W)
new_grid = torch.where(is_donor, new_grid * (1 - DON), new_grid)
new_grid

tensor([[[[4.5000, 0.5000, 0.0000],
          [0.0000, 9.0000, 1.0000],
          [0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.3000],
          [4.5000, 0.5000, 2.7000]]]])

In [46]:
empty_mask = (grid <= EPS).all(dim=1, keepdim=True)    # (B,1,H,W)
empty_mask

tensor([[[[False,  True,  True],
          [ True, False,  True],
          [False,  True, False]]]])

In [48]:
_, argmax_pop = new_grid.max(dim=1)  
one_hot = torch.nn.functional.one_hot(argmax_pop, num_classes=P) \
                    .permute(0,3,1,2).bool()

torch.where(one_hot & empty_mask, contrib, 0)

tensor([[[[0.0000, 0.5000, 0.0000],
          [0.0000, 0.0000, 1.0000],
          [0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000],
          [0.0000, 0.5000, 0.0000]]]])

In [ ]:
new_grid = torch.where(empty_mask, )